In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch_geometric.loader import DataLoader as GeoDataLoader
from torch_geometric.nn import global_mean_pool
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader


c:\Users\Admin\Desktop\ai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
import pickle
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import numpy as np
import pandas as pd
from numpy import savetxt, loadtxt
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score
from torch.nn import Linear, BatchNorm1d
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Input, Concatenate, Reshape, Conv1D, Flatten, Dense, Dropout, MultiHeadAttention, LayerNormalization, Add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
from transformers import AutoTokenizer, AutoModel

from tqdm import tqdm
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dropout, Dense, concatenate, BatchNormalization



In [3]:
def load_dataset(label_count):

    data = pd.read_csv("DATASET/multilabel_BILSTM_BERT.csv", index_col=False)
    data = data.drop(columns='Unnamed: 0')
    print(len(data))

    label_columns = ['Other', 'access_control', 'arithmetic', 'denial_service',
                     'front_running', 'reentrancy', 'time_manipulation',
                     'unchecked_low_calls']

    if label_count > len(label_columns):
        raise ValueError("Label count exceeds the available labels")

    selected_labels = label_columns[:label_count]

    for column in label_columns:
        data[column] = data[column].apply(lambda x: 1 if x != 0 else 0)

    # Giữ đúng 39645 rows
    #data = data.iloc[:39645]
    total_rows = len(data)
    split_point = int(0.8 * total_rows)

    print(split_point)
    print(total_rows - split_point)

    df_train = data.iloc[:split_point]
    df_test = data.iloc[split_point:]
    

    if True:
        X_train_source = np.load("DATASET/X_train_source_minmaxsummean_10chunks_512.npy")
        X_test_source = np.load("DATASET/X_test_source_minmaxsummean_10chunks_512.npy")
        X_train_source = X_train_source.reshape(-1, 10, 4, 768)
        X_test_source = X_test_source.reshape(-1, 10, 4, 768)
        X_train_source = X_train_source[:, :, 0, :]
        X_test_source = X_test_source[:, :, 0, :]
        X_train_source = X_train_source.reshape(-1, 10 * 768)
        X_test_source = X_test_source.reshape(-1, 10 * 768)

    else:

        # =========================
        # CODEBERT EMBEDDING
        # =========================

        source_train = df_train['sourcecode'].tolist()
        source_test = df_test['sourcecode'].tolist()
        

        tokenizer = AutoTokenizer.from_pretrained("./codebert")
        model = AutoModel.from_pretrained("./codebert")

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = model.to(device)


        model.eval()

        print("Generating CodeBERT embeddings...")

        X_train_source = get_codebert_embeddings(source_train, tokenizer, model, device)
        X_test_source = get_codebert_embeddings(source_test, tokenizer, model, device)
        
            
        # texts = df_train['sourcecode'].astype(str).tolist()

        # lengths = [len(tokenizer.tokenize(x)) for x in texts]

        # print("Average tokens:", np.mean(lengths))
        # print("Max tokens:", np.max(lengths))


        print("Load Features CodeBERT success!!")
        np.save("DATASET/X_train_source.npy", X_train_source)
        np.save("DATASET/X_test_source.npy", X_test_source)

    # =========================
    # OPCODE FEATURES (giữ nguyên)
    # =========================

    X_train_opcode = np.array(df_train.iloc[:, 11:].values)
    X_test_opcode = np.array(df_test.iloc[:, 11:].values)

    # =========================
    # LABELS
    # =========================

    y_train = df_train[selected_labels].values
    y_test = df_test[selected_labels].values
    labels = data[selected_labels].values

    print(f"Load Data for {label_count}-Label MultiLabel success!!")

    return X_train_opcode, X_test_opcode, X_train_source, X_test_source, y_train, y_test, labels

In [4]:
X_train_opcode, X_test_opcode, X_train_source, X_test_source, y_train, y_test, labels = load_dataset(8)

45597
36477
9120
Load Data for 8-Label MultiLabel success!!


In [5]:
import torch
import pickle
import numpy as np
import torch.nn.functional as F

from torch.nn import Linear, BatchNorm1d
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool
from sklearn.metrics import precision_score, recall_score, f1_score

# =========================
# LOAD DATA
# =========================
input_file = 'DATASET/gnn_input.pkl'

with open(input_file, 'rb') as f:
    raw_data = pickle.load(f)

feature_list = raw_data['features']
adj_matrices = raw_data['adj_matrices']
label_list = raw_data['labels']

target_dim = 57
num_classes = len(label_list[0])

def pad_or_truncate_features(features, target_dim):
    if features.shape[1] > target_dim:
        return features[:, :target_dim]
    elif features.shape[1] < target_dim:
        padded_features = np.zeros((features.shape[0], target_dim), dtype=np.float32)
        padded_features[:, :features.shape[1]] = features
        return padded_features
    else:
        return features.astype(np.float32)

def create_pyg_data(features, adj_matrix, label, target_dim):
    adj = torch.tensor(np.array(adj_matrix), dtype=torch.long)
    edge_index = adj.nonzero(as_tuple=False).t().contiguous()

    x = torch.tensor(pad_or_truncate_features(features, target_dim), dtype=torch.float)
    y = torch.tensor(label, dtype=torch.float)

    return Data(x=x, edge_index=edge_index, y=y)

data_list = [
    create_pyg_data(feat, adj, label, target_dim)
    for feat, adj, label in zip(feature_list, adj_matrices, label_list)
]

# nếu muốn shuffle trước khi split thì mở 2 dòng dưới
# indices = np.random.permutation(len(data_list))
# data_list = [data_list[i] for i in indices]

split_idx = int(len(data_list) * 0.8)
cfg_train_loader = DataLoader(data_list[:split_idx], batch_size=64, shuffle=False)
cfg_test_loader = DataLoader(data_list[split_idx:], batch_size=64, shuffle=False)


In [6]:

class OpcodeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)     # input cho Embedding
        self.y = torch.tensor(y, dtype=torch.float32)  # cho BCELoss

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
class SourceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [7]:
opcode_train_dataset = OpcodeDataset(X_train_opcode, y_train)
opcode_test_dataset = OpcodeDataset(X_test_opcode, y_test)

opcode_train_loader = DataLoader(opcode_train_dataset, batch_size=64, shuffle=False)
opcode_test_loader  = DataLoader(opcode_test_dataset, batch_size=64, shuffle=False)

source_train_dataset = SourceDataset(X_train_source, y_train)
source_test_dataset  = SourceDataset(X_test_source, y_test)

source_train_loader = DataLoader(source_train_dataset, batch_size=64, shuffle=False)
source_test_loader  = DataLoader(source_test_dataset, batch_size=64, shuffle=False)

In [8]:

class BiLSTMModel(nn.Module):
    def __init__(
        self,
        vocab_size=34000,
        embedding_dim=256,
        hidden_dim=128,
        output_dim=8
    ):
        super(BiLSTMModel, self).__init__()
        
        # Embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Stacked BiLSTM (2 layers)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=2,              # 🔥 2 layer BiLSTM
            batch_first=True,
            bidirectional=True,
            dropout=0.3               # dropout giữa các layer LSTM
        )
        
        # Regularization
        self.dropout = nn.Dropout(0.3)
        self.bn1 = nn.BatchNorm1d(hidden_dim * 2)
        
        # Dense layers
        self.fc1 = nn.Linear(hidden_dim * 2, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc2 = nn.Linear(128, 64)
        
        # Output
        self.out = nn.Linear(64, output_dim)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (B, 280)
        x = self.embedding(x)  # -> (B, 280, 286)
        
        # LSTM
        _, (h_n, _) = self.lstm(x)
        
        # Lấy hidden state cuối của 2 chiều (forward + backward)
        x = torch.cat((h_n[-2], h_n[-1]), dim=1)  # (B, 256)
        
        # Dense
        x = self.dropout(x)
        x = self.bn1(x)
        x = self.dropout(x)
        
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.bn2(x)
        x = self.dropout(x)
        
        x = self.fc2(x)
        x = torch.relu(x)
        return x
        
        x = self.out(x)
        x = self.sigmoid(x)
        
        return x

import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool

class GAT(nn.Module):
    def __init__(
        self,
        in_channels=57,
        hidden_channels=64,
        num_classes=8
    ):
        super(GAT, self).__init__()

        # ===== GAT Layers =====
        self.conv1 = GATConv(
            in_channels=in_channels,
            out_channels=hidden_channels,
            heads=4,
            dropout=0.3
        )

        self.conv2 = GATConv(
            in_channels=hidden_channels * 4,
            out_channels=hidden_channels,
            heads=2,
            dropout=0.3
        )

        # ===== Batch Normalization =====
        self.bn1 = nn.BatchNorm1d(hidden_channels * 4)
        self.bn2 = nn.BatchNorm1d(hidden_channels * 2)

        # ===== Dense Layers =====
        self.fc1 = nn.Linear(hidden_channels * 2, hidden_channels)

        # ===== Output Layer =====
        self.fc2 = nn.Linear(hidden_channels, num_classes)

        self.dropout = 0.3

    def forward(self, data):

        # ===== Inputs =====
        x = data.x
        edge_index = data.edge_index
        batch = data.batch

        # ===== GAT Layer 1 =====
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        # ===== GAT Layer 2 =====
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        # ===== Global Pooling =====
        x = global_mean_pool(x, batch)

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        # ===== Dense Layer =====
        x = self.fc1(x)
        x = F.relu(x)
        return x

        # ===== Output =====
        x = self.fc2(x)
        x = torch.sigmoid(x)

        return x

In [9]:
codebert_model = load_model('DATASET/bert.h5')
codebert_model = Model(
    inputs=codebert_model.input,
    outputs=codebert_model.layers[-2].output   # layer 64
)

bilstm_model = BiLSTMModel()
bilstm_model.load_state_dict(torch.load("DATASET/bilstm_model.pth"))
device = "cuda" if torch.cuda.is_available() else "cpu"
bilstm_model.to(device)
bilstm_model.eval()

gat_model = GAT()
gat_model.load_state_dict(torch.load("DATASET/gat.pth"))
gat_model.to(device)
gat_model.eval()

C:\Users\Admin\AppData\Local\Temp\ipykernel_21056\1889956571.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  bilstm_model.load_state_dict(torch.load("DATASET/bilstm_mode

GAT(
  (conv1): GATConv(57, 64, heads=4)
  (conv2): GATConv(256, 64, heads=2)
  (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=8, bias=True)
)

In [10]:
for i, layer in enumerate(codebert_model.layers):
    try:
        print(i, layer.name, layer.output_shape)
    except:
        print(i, layer.name, "no shape")

0 sourcecode_features no shape
1 batch_normalization_3 no shape
2 dense_6 no shape
3 dropout_4 no shape
4 dense_7 no shape
5 batch_normalization_4 no shape
6 dropout_5 no shape
7 dense_8 no shape
8 dropout_6 no shape
9 dense_9 no shape
10 batch_normalization_5 no shape
11 dropout_7 no shape
12 dense_10 no shape


In [11]:
# class MetaClassifier(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(192, 128),
#             nn.ReLU(),
#             nn.Dropout(0.3),

#             nn.Linear(128, 64),
#             nn.ReLU(),

#             nn.Linear(64, 8),
#             nn.Sigmoid()
#         )

#     def forward(self, x):
#         return self.net(x)

In [12]:
# class MetaClassifier(nn.Module):
#     def __init__(self):
#         super().__init__()
#         self.net = nn.Sequential(

#             nn.Linear(192, 8),
#             nn.Sigmoid()
#         )

#     def forward(self, x):
#         return self.net(x)
    
import torch
import torch.nn as nn


class MetaClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(

            # ===== Fusion Layer 1 =====
            nn.Linear(192, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            # ===== Fusion Layer 2 =====
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            # ===== Output =====
            nn.Linear(64, 8),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# import torch
# import torch.nn as nn


# class MetaClassifier(nn.Module):
#     def __init__(self):
#         super().__init__()

#         # ===== Conv1D Fusion =====
#         self.conv1d = nn.Conv1d(
#             in_channels=1,
#             out_channels=64,
#             kernel_size=3
#         )

#         # ===== Pooling =====
#         self.global_pool = nn.AdaptiveMaxPool1d(1)

#         # ===== Regularization =====
#         self.dropout = nn.Dropout(0.3)

#         # ===== Dense Layers =====
#         self.fc1 = nn.Linear(64, 64)

#         # ===== Output =====
#         self.out = nn.Linear(64, 8)

#     def forward(self, x):

#         # x shape:
#         # (batch_size, 192)

#         # ===== Reshape for Conv1D =====
#         x = x.unsqueeze(1)

#         # (batch, 1, 192)

#         # ===== Conv1D =====
#         x = self.conv1d(x)

#         # (batch, 64, 190)

#         x = torch.relu(x)

#         # ===== Global Max Pooling =====
#         x = self.global_pool(x)

#         # (batch, 64, 1)

#         x = x.squeeze(-1)

#         # (batch, 64)

#         # ===== Dropout =====
#         x = self.dropout(x)

#         # ===== Dense Layer =====
#         x = self.fc1(x)

#         x = torch.relu(x)

#         # ===== Output =====
#         x = self.out(x)

#         x = torch.sigmoid(x)

#         return x

In [13]:
def extract_opcode_features(model, loader, device):
    model.eval()
    outputs = []

    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            out = model(x)   # (B, 8)
            outputs.append(out.cpu())

    return torch.cat(outputs, dim=0)  # (N, 8)


def extract_gat_features(model, loader, device):
    model.eval()
    outputs = []

    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data)   # (B, 8)
            outputs.append(out.cpu())

    return torch.cat(outputs, dim=0)


codebert_features_train = codebert_model.predict(X_train_source, batch_size=64)
codebert_features_test  = codebert_model.predict(X_test_source, batch_size=64)

opcode_feat_train = extract_opcode_features(bilstm_model, opcode_train_loader, device)
opcode_feat_test  = extract_opcode_features(bilstm_model, opcode_test_loader, device)

gat_feat_train = extract_gat_features(gat_model, cfg_train_loader, device)
gat_feat_test  = extract_gat_features(gat_model, cfg_test_loader, device)

# CodeBERT (Keras)
codebert_feat_train = codebert_model.predict(X_train_source, batch_size=64)
codebert_feat_test  = codebert_model.predict(X_test_source, batch_size=64)

# convert numpy -> torch nếu cần
codebert_feat_train = torch.tensor(codebert_feat_train)
codebert_feat_test  = torch.tensor(codebert_feat_test)

train_features = torch.cat([
    opcode_feat_train,
    gat_feat_train,
    codebert_feat_train
], dim=1)   # (N, 24)

test_features = torch.cat([
    opcode_feat_test,
    gat_feat_test,
    codebert_feat_test
], dim=1)

570/570 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step
143/143 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
570/570 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step
143/143 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step


In [14]:
class FusionDataset(Dataset):
    def __init__(self, X, y):
        self.X = X.float()
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [15]:
train_dataset = FusionDataset(train_features, y_train)
test_dataset  = FusionDataset(test_features, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

meta_model = MetaClassifier().to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(meta_model.parameters(), lr=1e-3)

In [17]:
def train_model(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for X, y in train_loader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(X)

        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

from sklearn.metrics import classification_report
import numpy as np
import torch

def evaluate_model(model, test_loader, device):
    model.eval()

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for X, y in test_loader:
            X = X.to(device)

            outputs = model(X)  # (B, 8)

            all_preds.append(outputs.cpu())
            all_targets.append(y)

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    # Nếu dùng BCEWithLogitsLoss thì mở dòng này:
    # all_preds = 1 / (1 + np.exp(-all_preds))

    preds_bin = (all_preds > 0.5).astype(int)

    label_names = [
        'Other', 'access_control', 'arithmetic', 'denial_service',
        'front_running', 'reentrancy', 'time_manipulation',
        'unchecked_low_calls'
    ]

    print("\n===== CLASSIFICATION REPORT =====")
    print(classification_report(all_targets, preds_bin, target_names=label_names, digits=4))

In [18]:
EPOCHS = 2000

for epoch in range(EPOCHS):
    loss = train_model(meta_model, train_loader, optimizer, criterion, device)


    if (epoch + 1) % 100 == 0:
            
        print(f"\nEpoch {epoch+1}/{EPOCHS} - ", end='')
        print(f"Loss: {loss:.4f}")
        evaluate_model(meta_model, test_loader, device)


Epoch 100/2000 - Loss: 0.0312

===== CLASSIFICATION REPORT =====
                     precision    recall  f1-score   support

              Other     0.9410    0.9314    0.9361      6091
     access_control     0.7110    0.5840    0.6413       476
         arithmetic     0.9502    0.9658    0.9579      5755
     denial_service     0.8932    0.8967    0.8949      1268
      front_running     0.7883    0.7279    0.7569      1018
         reentrancy     0.9351    0.8731    0.9031      3798
  time_manipulation     0.8310    0.7838    0.8067       458
unchecked_low_calls     0.9552    0.9388    0.9469      4039

          micro avg     0.9297    0.9105    0.9200     22903
          macro avg     0.8756    0.8377    0.8555     22903
       weighted avg     0.9284    0.9105    0.9191     22903
        samples avg     0.8787    0.8660    0.8620     22903



c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



Epoch 200/2000 - Loss: 0.0296

===== CLASSIFICATION REPORT =====
                     precision    recall  f1-score   support

              Other     0.9387    0.9350    0.9368      6091
     access_control     0.7080    0.5756    0.6350       476
         arithmetic     0.9489    0.9672    0.9579      5755
     denial_service     0.9050    0.8943    0.8996      1268
      front_running     0.8002    0.7279    0.7623      1018
         reentrancy     0.9324    0.8710    0.9006      3798
  time_manipulation     0.8314    0.7860    0.8081       458
unchecked_low_calls     0.9507    0.9448    0.9477      4039

          micro avg     0.9289    0.9123    0.9205     22903
          macro avg     0.8769    0.8377    0.8560     22903
       weighted avg     0.9273    0.9123    0.9194     22903
        samples avg     0.8812    0.8692    0.8646     22903



c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



Epoch 300/2000 - Loss: 0.0267

===== CLASSIFICATION REPORT =====
                     precision    recall  f1-score   support

              Other     0.9435    0.9302    0.9368      6091
     access_control     0.6983    0.5882    0.6385       476
         arithmetic     0.9494    0.9679    0.9585      5755
     denial_service     0.9149    0.8817    0.8980      1268
      front_running     0.7978    0.7289    0.7618      1018
         reentrancy     0.9397    0.8665    0.9016      3798
  time_manipulation     0.8404    0.7817    0.8100       458
unchecked_low_calls     0.9553    0.9460    0.9506      4039

          micro avg     0.9326    0.9102    0.9212     22903
          macro avg     0.8799    0.8364    0.8570     22903
       weighted avg     0.9312    0.9102    0.9202     22903
        samples avg     0.8824    0.8664    0.8640     22903



...

c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


                     precision    recall  f1-score   support

              Other     0.9355    0.9353    0.9354      6091
     access_control     0.7215    0.5714    0.6377       476
         arithmetic     0.9463    0.9713    0.9587      5755
     denial_service     0.9154    0.8793    0.8970      1268
      front_running     0.7913    0.7151    0.7513      1018
         reentrancy     0.9381    0.8655    0.9003      3798
  time_manipulation     0.8125    0.7948    0.8035       458
unchecked_low_calls     0.9544    0.9438    0.9491      4039

          micro avg     0.9291    0.9110    0.9200     22903
          macro avg     0.8769    0.8346    0.8541     22903
       weighted avg     0.9276    0.9110    0.9187     22903
        samples avg     0.8794    0.8672    0.8628     22903


Epoch 900/2000 - Loss: 0.0243

===== CLASSIFICATION REPORT =====


c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\ai_env\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


                     precision    recall  f1-score   support

              Other     0.9337    0.9358    0.9347      6091
     access_control     0.7132    0.5693    0.6332       476
         arithmetic     0.9493    0.9689    0.9590      5755
     denial_service     0.9150    0.8825    0.8984      1268
      front_running     0.7904    0.7259    0.7568      1018
         reentrancy     0.9347    0.8665    0.8993      3798
  time_manipulation     0.8288    0.7926    0.8103       458
unchecked_low_calls     0.9577    0.9411    0.9493      4039

          micro avg     0.9294    0.9108    0.9200     22903
          macro avg     0.8778    0.8353    0.8551     22903
       weighted avg     0.9279    0.9108    0.9188     22903
        samples avg     0.8801    0.8673    0.8629     22903



KeyboardInterrupt: 